In [ ]:
#load SAE weights for all 8 trainers (4 SAE types x 2 regularization settings)
import torch
import os
from pathlib import Path


def _repo_root() -> Path:
    """Repo root containing downloaded artifact directories."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "data_model_weights").is_dir() or (cand / "scripts" / "download_artifact.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root. Run this notebook from inside the repository.")


base_path = str(_repo_root() / "data_model_weights")

# Define trainers to compare (no_reg, with_reg)
l2_trainers = ["trainer_12", "trainer_16"]       # TopK with L2
l1_trainers = ["trainer_6", "trainer_9"]         # TopK with L1
matryoshka_trainers = ["trainer_5", "trainer_6"] # Matryoshka with L1
batch_topk_trainers = ["trainer_6", "trainer_7"] # Batch TopK with L1

# Build paths for each trainer with descriptive names
trainer_paths = {
    # TopK L2
    "TopK-L2 (no reg)": os.path.join(base_path, "models_topk_l2", "resid_post_layer_3", "trainer_12", "ae.pt"),
    "TopK-L2 (with reg)": os.path.join(base_path, "models_topk_l2", "resid_post_layer_3", "trainer_16", "ae.pt"),
    # TopK L1
    "TopK-L1 (no reg)": os.path.join(base_path, "models_SAEBenchingTopK", "resid_post_layer_3", "trainer_6", "ae.pt"),
    "TopK-L1 (with reg)": os.path.join(base_path, "models_SAEBenchingTopK", "resid_post_layer_3", "trainer_9", "ae.pt"),
    # Matryoshka L1
    "Matryoshka-L1 (no reg)": os.path.join(base_path, "models_matryoshka_l1", "resid_post_layer_3", "trainer_5", "ae.pt"),
    "Matryoshka-L1 (with reg)": os.path.join(base_path, "models_matryoshka_l1", "resid_post_layer_3", "trainer_6", "ae.pt"),
    # Batch TopK L1
    "BatchTopK-L1 (no reg)": os.path.join(base_path, "models_SAEBenchingBatchTopK", "resid_post_layer_3", "trainer_6", "ae.pt"),
    "BatchTopK-L1 (with reg)": os.path.join(base_path, "models_SAEBenchingBatchTopK", "resid_post_layer_3", "trainer_7", "ae.pt"),
    # Matryoshka L2
    "Matryoshka-L2 (no reg)": os.path.join(base_path, "sae_models_matryoshka_L2", "resid_post_layer_3", "trainer_5", "ae.pt"),
    "Matryoshka-L2 (with reg)": os.path.join(base_path, "sae_models_matryoshka_L2", "resid_post_layer_3", "trainer_6", "ae.pt"),
    # Batch TopK L2
    "BatchTopK-L2 (no reg)": os.path.join(base_path, "sae_models_batch_topK_L2", "resid_post_layer_3", "trainer_5", "ae.pt"),
    "BatchTopK-L2 (with reg)": os.path.join(base_path, "sae_models_batch_topK_L2", "resid_post_layer_3", "trainer_7", "ae.pt"),
}

# Helper functions to get encoder/decoder weights regardless of naming convention
def get_encoder_weight(weights):
    """Get encoder weights - handles different naming conventions."""
    if 'encoder.weight' in weights:
        return weights['encoder.weight']
    elif 'W_enc' in weights:
        return weights['W_enc']
    else:
        raise KeyError(f"Could not find encoder weights. Keys: {weights.keys()}")

def get_decoder_weight(weights):
    """Get decoder weights - handles different naming conventions."""
    if 'decoder.weight' in weights:
        return weights['decoder.weight']
    elif 'W_dec' in weights:
        return weights['W_dec']
    else:
        raise KeyError(f"Could not find decoder weights. Keys: {weights.keys()}")

# Load all weights
all_weights = {}
for name, path in trainer_paths.items():
    print(f"Loading {name} from {path}")
    all_weights[name] = torch.load(path)
    print(f"  Keys: {all_weights[name].keys()}")
    enc_w = get_encoder_weight(all_weights[name])
    dec_w = get_decoder_weight(all_weights[name])
    print(f"  Encoder shape: {enc_w.shape}")
    print(f"  Decoder shape: {dec_w.shape}")


In [ ]:
# Summary of loaded weights
print("=" * 60)
print("Summary of all loaded SAE weights:")
print("=" * 60)
for name, weights in all_weights.items():
    print(f"\n{name}:")
    enc_w = get_encoder_weight(weights)
    dec_w = get_decoder_weight(weights)
    print(f"  Encoder shape: {enc_w.shape}")
    print(f"  Decoder shape: {dec_w.shape}")
    if 'k' in weights:
        print(f"  k (TopK): {weights['k']}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def compute_cosine_similarities(weights, verbose=False):
    """Compute cosine similarity between encoder and decoder weights for each feature.
    
    Handles different weight layouts:
    - TopK/BatchTopK: Encoder [n_features, d_model], Decoder [d_model, n_features]
    - Matryoshka: Encoder [d_model, n_features], Decoder [n_features, d_model]
    """
    We = get_encoder_weight(weights).data.cpu().numpy()
    Wd = get_decoder_weight(weights).data.cpu().numpy()
    
    # Detect layout based on shapes
    # TopK: Encoder [n_features, d_model], Decoder [d_model, n_features] → n_feats = We.shape[0]
    # Matryoshka: Encoder [d_model, n_features], Decoder [n_features, d_model] → n_feats = We.shape[1]
    
    if We.shape[0] > We.shape[1]:
        # TopK layout: Encoder [n_features, d_model]
        n_feats = We.shape[0]
        is_transposed = False
    else:
        # Matryoshka layout: Encoder [d_model, n_features]
        n_feats = We.shape[1]
        is_transposed = True
    
    if verbose:
        print(f"  Weight layout: {'Matryoshka (transposed)' if is_transposed else 'TopK (standard)'}")
    
    cos_sims = []
    zero_norm_count = 0
    
    for i in range(n_feats):
        if is_transposed:
            # Matryoshka: Encoder[:, i] and Decoder[i, :]
            enc_vec = We[:, i]
            dec_vec = Wd[i, :]
        else:
            # TopK: Encoder[i, :] and Decoder[:, i]
            enc_vec = We[i, :]
            dec_vec = Wd[:, i]
        
        norm_e = np.linalg.norm(enc_vec)
        norm_d = np.linalg.norm(dec_vec)
        
        if norm_e == 0 or norm_d == 0:
            cs = 0.0
            zero_norm_count += 1
        else:
            cs = np.dot(enc_vec, dec_vec) / (norm_e * norm_d)
        cos_sims.append(cs)
    
    if verbose:
        print(f"  Total features: {n_feats}")
        print(f"  Zero norm features: {zero_norm_count}")
        print(f"  Non-zero features: {n_feats - zero_norm_count}")
    
    return np.array(cos_sims)

# Compute cosine similarities for all trainers
all_cos_sims = {}
for name, weights in all_weights.items():
    print(f"\nComputing cosine similarities for {name}:")
    all_cos_sims[name] = compute_cosine_similarities(weights, verbose=True)
    
    # Summary statistics for non-zero features
    non_zero_sims = all_cos_sims[name][all_cos_sims[name] != 0]
    if len(non_zero_sims) > 0:
        print(f"  Mean (non-zero): {np.mean(non_zero_sims):.4f}")
        print(f"  Std (non-zero): {np.std(non_zero_sims):.4f}")
        print(f"  Min (non-zero): {np.min(non_zero_sims):.4f}")
        print(f"  Max (non-zero): {np.max(non_zero_sims):.4f}")
        print(f"  Features with cos_sim > 0.95: {np.sum(non_zero_sims > 0.95)}")

In [ ]:
# ============================================================
# Analyze and plot zero cosine similarities (dead features)
# ============================================================

import pandas as pd

# Count zero cosine similarities for each trainer
zero_counts = {}
total_features = {}
for name, cos_sims in all_cos_sims.items():
    zero_count = np.sum(cos_sims == 0)
    zero_counts[name] = zero_count
    total_features[name] = len(cos_sims)

# Create a dataframe for easier analysis
df_zeros = pd.DataFrame({
    'SAE Type': [name.split(' (')[0] for name in zero_counts.keys()],
    'Regularization': ['with reg' if 'with reg' in name else 'no reg' for name in zero_counts.keys()],
    'Zero Count': list(zero_counts.values()),
    'Total Features': list(total_features.values()),
    'Trainer': list(zero_counts.keys())
})
df_zeros['Zero Percentage'] = (df_zeros['Zero Count'] / df_zeros['Total Features'] * 100).round(2)

print("=" * 70)
print("Dead Features Analysis (Zero Cosine Similarity = Zero Norm Features)")
print("=" * 70)
print(df_zeros.to_string(index=False))
print()

# Bar chart: Dead features by SAE type (grouped by regularization)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Colors matching the main plots
sae_colors = {
    'TopK-L2': '#1f77b4',
    'TopK-L1': '#ff7f0e',
    'Matryoshka-L1': '#2ca02c',
    'BatchTopK-L1': '#d62728',
    'Matryoshka-L2': '#9467bd',
    'BatchTopK-L2': '#8c564b',
}

# Left: Absolute count of dead features
ax1 = axes[0]
sae_types_list = list(sae_colors.keys())
x = np.arange(len(sae_types_list))
width = 0.35

no_reg_counts = [df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'no reg')]['Zero Count'].values[0] 
                 if len(df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'no reg')]) > 0 else 0 
                 for sae in sae_types_list]
with_reg_counts = [df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'with reg')]['Zero Count'].values[0] 
                   if len(df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'with reg')]) > 0 else 0 
                   for sae in sae_types_list]

bars1 = ax1.bar(x - width/2, no_reg_counts, width, label='No Regularization', alpha=0.8, color=[sae_colors[s] for s in sae_types_list])
bars2 = ax1.bar(x + width/2, with_reg_counts, width, label='With Regularization', alpha=0.8, 
                color=[sae_colors[s] for s in sae_types_list], hatch='///')

ax1.set_xlabel('SAE Type', fontsize=12)
ax1.set_ylabel('Number of Dead Features', fontsize=12)
ax1.set_title('Dead Features Count by SAE Type\n(Zero Cosine Similarity)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(sae_types_list, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    if bar.get_height() > 0:
        ax1.annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    if bar.get_height() > 0:
        ax1.annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

# Right: Percentage of dead features
ax2 = axes[1]
no_reg_pct = [df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'no reg')]['Zero Percentage'].values[0] 
              if len(df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'no reg')]) > 0 else 0 
              for sae in sae_types_list]
with_reg_pct = [df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'with reg')]['Zero Percentage'].values[0] 
                if len(df_zeros[(df_zeros['SAE Type'] == sae) & (df_zeros['Regularization'] == 'with reg')]) > 0 else 0 
                for sae in sae_types_list]

bars3 = ax2.bar(x - width/2, no_reg_pct, width, label='No Regularization', alpha=0.8, color=[sae_colors[s] for s in sae_types_list])
bars4 = ax2.bar(x + width/2, with_reg_pct, width, label='With Regularization', alpha=0.8, 
                color=[sae_colors[s] for s in sae_types_list], hatch='///')

ax2.set_xlabel('SAE Type', fontsize=12)
ax2.set_ylabel('Percentage of Dead Features (%)', fontsize=12)
ax2.set_title('Dead Features Percentage by SAE Type\n(Zero Cosine Similarity)', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(sae_types_list, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars3:
    if bar.get_height() > 0:
        ax2.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars4:
    if bar.get_height() > 0:
        ax2.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Histogram including zeros (for trainers with significant dead features)
trainers_with_dead = [name for name, count in zero_counts.items() if count > 0]
if trainers_with_dead:
    print(f"\nTrainers with dead features (zero cosine sim): {trainers_with_dead}")
    
    fig, axes = plt.subplots(1, len(trainers_with_dead), figsize=(6*len(trainers_with_dead), 5))
    if len(trainers_with_dead) == 1:
        axes = [axes]
    
    for idx, name in enumerate(trainers_with_dead):
        ax = axes[idx]
        cos_sims = all_cos_sims[name]
        
        # Plot histogram including zeros
        ax.hist(cos_sims, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label=f'Zero (n={zero_counts[name]})')
        
        ax.set_title(f'{name}\n(Including {zero_counts[name]} Dead Features)', fontsize=12, fontweight='bold')
        ax.set_xlabel('Cosine Similarity')
        ax.set_ylabel('Count')
        ax.legend()
        ax.grid(alpha=0.3)
    
    plt.suptitle('Cosine Similarity Distribution Including Dead Features (Zero)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("\nNo trainers have dead features (zero cosine similarity).")

In [ ]:
# Overlaid histogram for each SAE type: no reg vs with reg
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sae_types = [
    ("TopK-L2", "TopK-L2 (no reg)", "TopK-L2 (with reg)", '#1f77b4'),
    ("TopK-L1", "TopK-L1 (no reg)", "TopK-L1 (with reg)", '#ff7f0e'),
    ("Matryoshka-L1", "Matryoshka-L1 (no reg)", "Matryoshka-L1 (with reg)", '#2ca02c'),
    ("BatchTopK-L1", "BatchTopK-L1 (no reg)", "BatchTopK-L1 (with reg)", '#d62728'),
    ("Matryoshka-L2", "Matryoshka-L2 (no reg)", "Matryoshka-L2 (with reg)", '#9467bd'),
    ("BatchTopK-L2", "BatchTopK-L2 (no reg)", "BatchTopK-L2 (with reg)", '#8c564b'),
]

for idx, (sae_name, no_reg_key, with_reg_key, base_color) in enumerate(sae_types):
    ax = axes[idx // 3, idx % 3]
    
    # No reg - lighter color
    non_zero_no_reg = all_cos_sims[no_reg_key][all_cos_sims[no_reg_key] != 0]
    ax.hist(non_zero_no_reg, bins=50, alpha=0.5, label=f'{sae_name} (no reg)', density=True, color=base_color)
    
    # With reg - darker with hatching
    non_zero_with_reg = all_cos_sims[with_reg_key][all_cos_sims[with_reg_key] != 0]
    ax.hist(non_zero_with_reg, bins=50, alpha=0.5, label=f'{sae_name} (with reg)', density=True, 
            color=base_color, hatch='///', edgecolor='black')
    
    ax.set_title(f'{sae_name}: No Reg vs With Reg', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.set_xlim(-0.2, 1.1)
    ax.axvline(x=0.95, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Effect of Regularization on Encoder-Decoder Cosine Similarity\n(By SAE Type)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



